In [1]:
import pathlib, re, tifffile, numpy as np, pandas as pd
from liffile import LifFile
import matplotlib.pyplot as plt

In [2]:
def _step(axis, xa):
    if axis not in xa.coords or xa.coords[axis].size < 2:
        return None
    return float(xa.coords[axis][1] - xa.coords[axis][0])

In [ ]:

src_root = pathlib.Path(r"Z:\Bel\Farid_Homogeneity")
tif_root = pathlib.Path(r"Z:\Bel\Farid_Homogeneity\tifs")
tif_root.mkdir(parents=True, exist_ok=True)

def _safe(name):
    """Make a string safe to use in a filename."""
    return re.sub(r"[^\w.\-]+", "_", str(name)).strip("_")

def _save_imagej(path, stack4d, x_um, z_um):
    """Save a (T, Z, Y, X) stack as an ImageJ hyperstack with calibration."""
    resolution = (1.0 / x_um, 1.0 / x_um) if x_um else None
    metadata = {"axes": "TZYX", "unit": "um"}
    if z_um:
        metadata["spacing"] = z_um
    tifffile.imwrite(str(path), stack4d, imagej=True,
                     resolution=resolution, metadata=metadata)


lif_paths = sorted(src_root.rglob("*.lif"))
print(f"Found {len(lif_paths)} LIF files\n")

for lif_path in lif_paths:
    lif_stem = lif_path.stem
    with LifFile(lif_path) as lif:
        for img in lif.images:
            image_name = "".join(img.path)
            dims = tuple(img.dims)
            if "Z" not in dims:  # need 4D x,y,z,t
                continue


            xa = img.asxarray()
            x_um = _step("X", xa) * 1e6 if _step("X", xa) is not None else None
            z_um = _step("Z", xa) * 1e6 if _step("Z", xa) is not None else None

            base = f"{_safe(lif_stem)}__{_safe(image_name)}"
            print(f"{lif_path.name} :: {image_name}  shape={xa.shape}  x={x_um}µm z={z_um}µm")

            # 1) original
            _save_imagej(tif_root / f"{base}_original.tif", xa.values, x_um, z_um)

print("\nDone.")


Found 2 LIF files

24.07.26 N2 Reservoirs day 7.lif :: R 4 Wide_Merged  shape=(2, 36, 6511, 2869)  x=1.3µm z=9.99954µm
24.07.26 N2 Reservoirs day 7.lif :: R 3 Wide_Merged  shape=(2, 37, 6504, 2875)  x=1.3µm z=9.999538888888889µm
24.07.26 N2 Reservoirs day 7.lif :: R 2 Wide_Merged  shape=(2, 37, 6507, 2847)  x=1.3µm z=9.999538888888889µm
24.07.26 N2 Reservoirs day 7.lif :: R 1 wide_Merged  shape=(2, 40, 6516, 2893)  x=1.3µm z=9.999541025641026µm
24.07.26 N2 Reservoirs day 7.lif :: R 8 small_Merged  shape=(2, 40, 6339, 2883)  x=1.3µm z=9.999541025641026µm
24.07.26 N2 Reservoirs day 7.lif :: R 7 small_Merged  shape=(2, 35, 6522, 2869)  x=1.3µm z=9.999541176470588µm


In [ ]:
def tif_pixel_sizes(tif_path):
    with tifffile.TiffFile(tif_path) as tif:
        tif_tags = {}
        for tag in tif.pages[0].tags.values():
            name, value = tag.name, tag.value
            tif_tags[name] = value

        x_pixel_size_um = 1/((tif_tags["XResolution"])[0]/(tif_tags["XResolution"][1]))
        y_pixel_size_um = 1/((tif_tags["YResolution"])[0]/(tif_tags["YResolution"][1]))
        # try:
        #     z_pixel_size_um = float(str(tif_tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
        # except:
        #     z_pixel_size_um = (float(str(tif_tags["ImageDescription"]).split("spacing=")[1].split("loop")[0]))
        
    return [x_pixel_size_um,y_pixel_size_um]# ,z_pixel_size_um]


In [ ]:
tif_root = pathlib.Path(r"Z:\Bel\Farid_Homogeneity\tifs")
tif_paths = sorted(tif_root.rglob("*.tif"))
max_proj_root = pathlib.Path(r"Z:\Bel\Farid_Homogeneity\max_projections")
max_proj_root.mkdir(parents=True, exist_ok=True)
sum_proj_root = pathlib.Path(r"Z:\Bel\Farid_Homogeneity\sum_projections")
sum_proj_root.mkdir(parents=True, exist_ok=True)
for tif_path in tif_paths:
    [x_pixel_size_um,y_pixel_size_um] = tif_pixel_sizes(tif_path)
    image = tifffile.imread(tif_path)
    image_name = tif_path.stem
    max_proj = np.max(image, axis=1)  # max projection along Z
    sum_proj = np.sum(image, axis=1)  # sum projection along Z
    tifffile.imwrite(
        sum_proj_root / f"{image_name}_sum_proj.tif",
        sum_proj,
        resolution=(10000 / x_pixel_size_um, 10000 / y_pixel_size_um),
        resolutionunit="CENTIMETER",
        metadata={
            "unit": "um",
        },
    )
    tifffile.imwrite(
        max_proj_root / f"{image_name}_max_proj.tif",
        max_proj,
        resolution=(10000 / x_pixel_size_um, 10000 / y_pixel_size_um),
        resolutionunit="CENTIMETER",
        metadata={
            "unit": "um",
        },
    )
    